# Data quality report cho một bộ dữ liệu mới: Rossmann store sales

Bộ `store-sales`: doanh số ngày của 1 115 cửa hàng Rossmann (chuỗi dược mỹ phẩm Đức), 01/01/2013 tới 31/07/2015.
`train.csv` (1 017 209 dòng: `Store`, `DayOfWeek`, `Date`, `Sales`, `Customers`, `Open`, `Promo`, `StateHoliday`,
`SchoolHoliday`) và `store.csv` (1 115 dòng: loại cửa hàng, khoảng cách và thời điểm đối thủ mở, chương trình
`Promo2`).

Team BI chuẩn bị đưa bộ này vào dashboard doanh số tuần và nhờ bạn: **"Trước khi build, kiểm giúp bộ này có vấn đề gì,
cái nào chặn, cái nào chỉ cần ghi chú."** Sản phẩm là một **data quality report** một trang (`starter/REPORT.md`) mà
team BI và quản lý vùng đọc hiểu được, không phải một notebook đầy `df.info()`.

Phần được chấm tự động: hai hàm kiểm tái dùng được (`run_checks`, `coverage_gaps`) và khung báo cáo. Nội dung báo cáo
chấm theo rubric trong `SOLUTION.md`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/06-data-cleaning/exercises/advanced/01-store-sales-quality-report -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có (train.csv khoảng 38 MB)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

STORE = ensure("store-sales")
train = pd.read_csv(STORE / "train.csv", dtype={"StateHoliday": str}, parse_dates=["Date"])
store = pd.read_csv(STORE / "store.csv")
print(train.shape, store.shape)
print(train.dtypes)
store.head()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

## 1. Bộ kiểm khai báo được

Một DQ check tốt là **dữ liệu**, không phải đoạn code rải rác: khai báo một danh sách kiểm, chạy một hàm, ra một bảng.
Cùng danh sách đó chạy lại mỗi tuần (hoặc chuyển thành dbt tests, pandera schema, Great Expectations suite).

`run_checks(df, checks) -> DataFrame`, mỗi phần tử của `checks` là một dict có `name`, `kind`, tuỳ chọn `severity`
(mặc định `"error"`) và tham số theo `kind`:

| `kind` | Tham số | Dòng lỗi |
| --- | --- | --- |
| `not_null` | `column` | ô thiếu |
| `unique` | `columns` (list) | **mọi** dòng thuộc một tổ hợp khoá xuất hiện hơn một lần |
| `range` | `column`, `min`, `max` | giá trị có mặt và nằm ngoài `[min, max]` |
| `allowed` | `column`, `values` | giá trị có mặt và không thuộc `values` |
| `rule` | `rule` (biểu thức `df.eval` mô tả dòng **hợp lệ**) | dòng biểu thức cho `False`; kết quả thiếu coi là hợp lệ |

`kind` khác thì `raise ValueError`. Kết quả: một dòng mỗi check theo thứ tự đầu vào, cột `name`, `kind`, `severity`,
`failed` (số dòng lỗi, `int`), `failed_share` (trên số dòng của `df`), `passed` (`failed == 0`).

In [ ]:
def run_checks(df: pd.DataFrame, checks: list[dict]) -> pd.DataFrame:
    """Chạy danh sách check khai báo, trả bảng name, kind, severity, failed, failed_share, passed."""
    # TODO: với mỗi check, tạo mặt nạ dòng lỗi theo kind (ValueError nếu kind lạ)
    # TODO: ghép kết quả thành DataFrame đúng thứ tự cột
    raise NotImplementedError

## 2. Độ phủ theo thời gian

Kiểm từng dòng không phát hiện được **dòng không có mặt**. Với dữ liệu "mỗi cửa hàng mỗi ngày một dòng", kiểm xem mỗi
cửa hàng có đủ ngày không.

`coverage_gaps(df, key, date_col) -> DataFrame`: lịch đầy đủ là mọi ngày từ ngày nhỏ nhất tới lớn nhất của **cả bảng**.
Với mỗi giá trị `key` thiếu ít nhất một ngày: `missing_days`, `first_missing`, `last_missing`. Index tên `key`, sắp
`missing_days` giảm dần (bằng nhau giữ thứ tự khoá tăng dần). Khoá đủ ngày không có dòng.

In [ ]:
def coverage_gaps(df: pd.DataFrame, key: str, date_col: str) -> pd.DataFrame:
    """Số ngày thiếu, ngày thiếu đầu và cuối của mỗi khoá so với lịch đầy đủ của cả bảng."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: khai báo bộ kiểm cho train.csv. Thêm, bớt, đặt severity theo đánh giá của bạn.
TRAIN_CHECKS = [
    {"name": "key_unique", "kind": "unique", "columns": ["Store", "Date"]},
    {"name": "sales_not_null", "kind": "not_null", "column": "Sales"},
    {"name": "holiday_codes", "kind": "allowed", "column": "StateHoliday", "values": ["0", "a", "b", "c"]},
    {"name": "closed_no_sales", "kind": "rule", "rule": "not (Open == 0 and Sales > 0)"},
    {"name": "open_has_sales", "kind": "rule", "rule": "not (Open == 1 and Sales == 0)", "severity": "warn"},
    # TODO của bạn: Customers, DayOfWeek khớp Date, khoảng giá trị Sales hợp lý, ...
]
run_checks(train, TRAIN_CHECKS)

In [ ]:
# explore: bộ kiểm cho store.csv
STORE_CHECKS = [
    {"name": "store_unique", "kind": "unique", "columns": ["Store"]},
    {"name": "distance_not_null", "kind": "not_null", "column": "CompetitionDistance"},
    # TODO của bạn: năm đối thủ mở cửa hợp lý, Promo2 và Promo2Since nhất quán, ...
]
run_checks(store, STORE_CHECKS)

In [ ]:
# explore: độ phủ theo cửa hàng, và nó làm chart doanh số tháng nói gì
gaps = coverage_gaps(train, "Store", "Date")
print(len(gaps), "cửa hàng thiếu ngày;", gaps.groupby(["missing_days", "first_missing", "last_missing"]).size().to_dict())
monthly = train.groupby(train.Date.dt.to_period("M"))["Sales"].sum() / 1e6
stores = train.groupby(train.Date.dt.to_period("M"))["Store"].nunique()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(monthly.index.to_timestamp(), monthly, marker="o", ms=3)
ax2 = ax.twinx()
ax2.plot(stores.index.to_timestamp(), stores, color="#e45756", lw=1)
ax.set_ylabel("doanh số tháng (triệu)")
ax2.set_ylabel("số cửa hàng có dữ liệu", color="#e45756")
ax.set_title("TODO: viết headline sau khi đọc chart", loc="left")
plt.show()

## 3. Từ bảng kiểm tới báo cáo

Điều tra thêm theo hướng bạn thấy cần (thêm cell `explore`): ai bị ảnh hưởng, ảnh hưởng tới chỉ số nào trên dashboard,
vấn đề là **lỗi dữ liệu**, **quy ước của nguồn** hay **sự thật nghiệp vụ** (cửa hàng đóng cửa sửa chữa?). Rồi viết
`starter/REPORT.md` theo khung có sẵn: dòng đầu là headline có con số, 300-900 từ, mỗi vấn đề có mức độ (chặn / cần xử
lý / ghi chú), tác động và người cần làm.